# Actividad 1 · Mejora de imagen: operaciones elementales
Visión Artificial — MIA UNIR

Estructura del notebook:
1. Configuración y carga de las 4 imágenes de Dark Face
2. Utilidades: métricas de calidad y figuras antes/después con histograma
3. Técnicas: negativo, estiramiento lineal, log, gamma, ecualización, operaciones aritméticas y realce laplaciano
4. Tabla comparativa y exportación de figuras para la memoria

> Recuerda: este código va en los **anexos** de la memoria; en *Material y métodos* las técnicas se describen de forma formal, sin referencias a Python.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Todas las funciones (carga, métricas, figuras y técnicas) están en mejora_imagen.py
from mejora_imagen import (
    cargar_imgs_propias, metricas, sigma_ruido, recorte, comparar, Tecnica, Cadena,
    negativo, estiramiento_lineal, transformacion_log, correccion_gamma,
    ecualizacion_global,
    suma_constante, multiplicacion_constante, realce_laplaciano,
)

# Carpeta con las 4 imágenes de Dark Face y carpeta donde se guardan las figuras
DIR_IMGS = Path("imgs")
DIR_FIGS = Path("figures")
DIR_FIGS.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 110

## 1. Carga de imágenes
Dark Face son imágenes **en color** (JPG). Hay dos opciones, y la elección debes justificarla en la memoria:
- **Trabajar en escala de grises** (más simple, histogramas de un solo canal).
- **Trabajar sobre la luminancia** (canal Y de YCrCb o V de HSV) y reconstruir el color: evita que la ecualización por canales RGB altere los tonos.

`cargar_imagenes` devuelve ambas versiones para que puedas elegir.

In [ ]:
imagenes = cargar_imgs_propias(DIR_IMGS, [182, 5069, 5940, 5019])

# Características para la sección Material y métodos (resolución, profundidad, etc.)
pd.DataFrame([
    {"imagen": n, "alto": d["bgr"].shape[0], "ancho": d["bgr"].shape[1],
     "canales": d["bgr"].shape[2], "tipo": str(d["bgr"].dtype)}
    for n, d in imagenes.items()
])

## 2. Utilidades
Todas las funciones están en `mejora_imagen.py` (importadas en la primera celda).
### 2.1 Métricas de calidad
`metricas(img)`, sobre la imagen en grises (o la luminancia) en `uint8`:
- **Brillo medio** μ.
- **Contraste RMS**: desviación típica σ de las intensidades.
- **Contraste de Michelson** con percentiles 1 y 99 (más robusto que min/max ante píxeles aislados).
- **Entropía de Shannon** (bits): cuánta información/variedad de niveles hay.
- **% de píxeles saturados** a 0 o a 255: sirve para detectar artefactos (recortes) de algunas técnicas.

### 2.2 Figura antes/después
`comparar(imagenes_dict, ...)` muestra cada imagen con su histograma debajo. Parámetros principales:
- `filas` / `columnas`: forma de la rejilla (si solo das uno, el otro se calcula como ⌈n / ·⌉).
- `ancho_celda` en pulgadas por imagen (el alto sale de la proporción de la imagen), o `figsize` para fijar el tamaño total.
- `con_histograma`, `alto_histograma` (alto relativo a la imagen), `hist_log` (eje vertical logarítmico).
- `tam_letra` de los subtítulos, `titulo` general.
- `guardar=DIR_FIGS / "fichero.png"` y `dpi` para exportar; `mostrar=False` para retocar la figura antes de mostrarla (devuelve `fig, ejes_img, ejes_hist`).

### 2.3 Aplicar una técnica sobre la luminancia (opcional)
Si trabajas en color, `sobre_luminancia(bgr, f)` (en `mejora_imagen.py`) aplica cualquier transformación `f(uint8 2D) -> uint8 2D` solo al canal Y y conserva la crominancia.

### 2.4 Exploración inicial
Antes de aplicar nada, mira cada original y su histograma: ¿dónde se concentran los niveles? ¿hay zonas quemadas (farolas, faros)? ¿cuánto ruido se ve en las zonas oscuras? Esto justifica luego qué técnicas eliges y con qué parámetros.

In [ ]:
for nombre, d in imagenes.items():
    comparar({"Original (color)": d["bgr"], "Original (grises)": d["gris"]},
             titulo=nombre, guardar=DIR_FIGS / f"{nombre}_original.png")

In [ ]:
pd.DataFrame({n: metricas(d["gris"]) for n, d in imagenes.items()}).T.round(3)

## 3. Técnicas de mejora
Cada técnica es una función `uint8 -> uint8` definida en `mejora_imagen.py` (sección 3 del módulo), con su fórmula en el comentario o docstring.
Para cada una, piensa qué parámetros vas a probar y **por qué** (eso va a *Material y métodos*).

Pistas de equivalencias con MATLAB:
| MATLAB | Python |
|---|---|
| `imcomplement` | `255 - img` |
| `imadjust(I, stretchlim(I), [])` | `exposure.rescale_intensity(img, in_range=(p_inf, p_sup))` con percentiles |
| `imadjust(I, [], [], gamma)` | `exposure.adjust_gamma(img, gamma)` |
| log: `c*log(1+I)` | `exposure.adjust_log` o con NumPy |
| `histeq` | `cv2.equalizeHist` |
| `imadd`, `immultiply` | `cv2.add`, `cv2.multiply` (saturan a 255, igual que MATLAB) |

⚠️ Cuidado con los tipos: con `uint8`, `img + 50` en NumPy **desborda** (da la vuelta a 0). Usa `cv2.add` o convierte a `float` y recorta con `np.clip`.

### 3.1 Prueba de una técnica (ejemplo de uso)
Patrón a repetir con cada técnica y cada imagen. Prueba varios valores de parámetro y quédate con los representativos para la memoria.

In [ ]:
nombre = next(iter(imagenes))   # primera imagen; cambia por la que quieras
I = imagenes[nombre]["gris"]

comparar({"Original": I,
          "γ = 0.4": correccion_gamma(I, 0.4),
          "γ = 0.6": correccion_gamma(I, 0.6)},
         titulo=f"{nombre} · corrección gamma", guardar=DIR_FIGS / f"{nombre}_gamma.png")

### 3.2 Realce laplaciano
g = f − c·∇²(G_σ * f). Sobre las originales de Dark Face apenas hay transiciones que realzar (contraste muy bajo), así que se aplica también **después de aclarar** (gamma 0.5) para ver su efecto real.

Se comparan tres variantes sobre la imagen aclarada:
- σ = 0 (laplaciano puro), c = 1.
- σ = 1 (LoG), c = 1: mismo realce con suavizado previo del término de detalle.
- 8-vecindad, σ = 1, c = 1: núcleo isótropo en más direcciones (respuesta más fuerte).

Además de la figura completa se muestra un **recorte ampliado** (a tamaño completo el ruido no se aprecia) y una tabla con las métricas habituales más una estimación de la desviación típica del ruido σ̂ (Immerkær, 1996):

σ̂ = √(π/2) · 1/(6(W−2)(H−2)) · Σ |(f * N)(x,y)|,  con N = [1 −2 1; −2 4 −2; 1 −2 1]

N es la diferencia de dos laplacianos discretos, por lo que anula (casi) la estructura de la imagen y deja sobre todo el ruido.

In [ ]:
variantes_realce = {
    "c=1, σ=0":        lambda x: realce_laplaciano(x, c=1.0, vecindad=4, sigma=0.0),
    "c=1, σ=1":        lambda x: realce_laplaciano(x, c=1.0, vecindad=4, sigma=1.0),
    "c=1, σ=1, 8-vec": lambda x: realce_laplaciano(x, c=1.0, vecindad=8, sigma=1.0),
}

filas_realce = []
for nombre, d in imagenes.items():
    base = correccion_gamma(d["gris"], 0.5)          # aclarado previo
    salidas = {"Gamma 0.5": base,
               **{f"Gamma 0.5 + realce ({k})": f(base) for k, f in variantes_realce.items()}}

    comparar(salidas, titulo=f"{nombre} · realce laplaciano",
             guardar=DIR_FIGS / f"{nombre}_realce_laplaciano.png")
    # Recorte relativo: ajusta la posición a una zona con detalle y zonas oscuras
    comparar({k: recorte(v) for k, v in salidas.items()},
             titulo=f"{nombre} · realce laplaciano (recorte ampliado)",
             guardar=DIR_FIGS / f"{nombre}_realce_laplaciano_recorte.png")

    # También sobre la original, para comparar con aplicarlo tras el aclarado
    salidas_tabla = {"Original": d["gris"],
                     "Original + realce (c=1, σ=1)": variantes_realce["c=1, σ=1"](d["gris"]),
                     **salidas}
    for etiqueta, img in salidas_tabla.items():
        filas_realce.append({"imagen": nombre, "variante": etiqueta,
                             **metricas(img), "sigma_ruido": sigma_ruido(img)})

tabla_realce = pd.DataFrame(filas_realce).set_index(["imagen", "variante"]).round(3)
tabla_realce.to_csv(DIR_FIGS / "tabla_realce_laplaciano.csv")
tabla_realce

## 4. Tabla comparativa
Define aquí las técnicas con los parámetros finales elegidos. La tabla resultante (métricas por imagen y técnica) es la base de la sección *Resultados*.

In [ ]:
tecnicas = {
    "Original": lambda x: x,
    "Negativo": negativo,
    "Estiramiento lineal": estiramiento_lineal,
    "Log": transformacion_log,
    "Gamma 0.5": lambda x: correccion_gamma(x, 0.5),
    "Ecualización global": ecualizacion_global,
    "Suma +40": lambda x: suma_constante(x, 40),
    "Multiplicación ×2": lambda x: multiplicacion_constante(x, 2),
    "Realce laplaciano (c=1, σ=1)": lambda x: realce_laplaciano(x, c=1.0, sigma=1.0),
}

filas = []
for nombre, d in imagenes.items():
    for tecnica, f in tecnicas.items():
        salida = f(d["gris"])   # se calcula una vez y se usa para todas las métricas
        # sigma_ruido: desviación típica estimada del ruido (Immerkær, 1996)
        filas.append({"imagen": nombre, "técnica": tecnica,
                      **metricas(salida), "sigma_ruido": sigma_ruido(salida)})

tabla = pd.DataFrame(filas).set_index(["imagen", "técnica"]).round(3)
tabla.to_csv(DIR_FIGS / "tabla_metricas.csv")
tabla

### 4.1 Rejilla resumen por imagen
Una figura por imagen con todas las técnicas: útil para el cuerpo de la memoria (o los anexos si ocupa mucho).

In [ ]:
for nombre, d in imagenes.items():
    comparar({t: f(d["gris"]) for t, f in tecnicas.items()},
             titulo=nombre, filas=2, tam_letra=9,
             guardar=DIR_FIGS / f"{nombre}_resumen.png")

## 5. Ideas para la discusión
- ¿Qué técnica sube más la entropía y el contraste? ¿Coincide con la que *se ve* mejor?
- ¿Cuál amplifica más el **ruido** de las zonas oscuras? (Tema 4)
- ¿Cuál satura farolas o faros? Mira `%_blanco`.
- ¿Funciona igual en las 4 imágenes? Relaciónalo con la forma del histograma original.
- Mejoras posibles: combinar técnicas (p. ej. filtrado de ruido + ecualización), gamma adaptativa, trabajar en luminancia…

In [ ]:
import numpy as np
from mejora_imagen import Tecnica

In [ ]:
gammas = np.linspace(0.1, 1.1, 5)
gammas = gammas.tolist()

nombre = next(iter(imagenes))

In [ ]:
def comparativa_mult(tecnicas, nombre: str, sufijo: str, titulo: str):
    """Aplica cada Tecnica a la imagen `nombre`: rejilla con histogramas + tabla de métricas.

    Las etiquetas de la figura y de la tabla salen de str(t), así que coinciden
    siempre con el parámetro realmente aplicado.
    """
    I = imagenes[nombre]["gris"]
    
    dic = {"Original": I}
    # sigma_ruido: desviación típica estimada del ruido (Immerkær), además de las métricas habituales
    filas = [{"imagen": f"{nombre}", "técnica": "Original",  **metricas(I), "sigma_ruido": sigma_ruido(I)}]
    for t in tecnicas:
        I_t = t(I)
        dic[str(t)] = I_t
        filas.append({"imagen": f"{nombre}", "técnica": str(t),
                      **metricas(I_t), "sigma_ruido": sigma_ruido(I_t)})
    
    
    comparar(dic,
             titulo=f"{nombre} · {titulo}", guardar=DIR_FIGS / f"{nombre}_mult_{sufijo}.png", columnas=3, filas=2)
    
    tabla_comparativa = pd.DataFrame(filas).set_index(["imagen","técnica"]).round(3)
    tabla_comparativa.to_csv(DIR_FIGS / f"{nombre}_comparativa_mult_{sufijo}.csv")
    print(tabla_comparativa)


def gamma_comparativa_mult(gammas, nombre: str):
    tecnicas = [Tecnica(correccion_gamma, "Gamma", {"γ": g}) for g in gammas]
    comparativa_mult(tecnicas, nombre, sufijo="gamma", titulo="corrección gamma")

In [ ]:
gamma_comparativa_mult(gammas, nombre)

In [ ]:
nombres = ["182", "5069", "5940", "5019"]

gammas = np.linspace(1.25, 2, 5)
gammas = gammas.tolist()

In [ ]:
gamma_comparativa_mult(gammas=gammas, nombre=nombres[1])

In [ ]:
def multiplicacion_comparativa_mult(ks, nombre: str):
    tecnicas = [Tecnica(multiplicacion_constante, "Multiplicación", {"k": k}) for k in ks]
    comparativa_mult(tecnicas, nombre, sufijo="multiplicacion", titulo="multiplicación por constante")

In [ ]:
multiplicacion_comparativa_mult(np.linspace(2, 8, 7).tolist(), nombre=nombres[0])

In [ ]:
"""from mejora_imagen import (
    cargar_imgs_propias, metricas, sigma_ruido, recorte, comparar, Tecnica,
    negativo, estiramiento_lineal, transformacion_log, correccion_gamma,
    ecualizacion_global,
    suma_constante, multiplicacion_constante, realce_laplaciano,
)"""